<a href="https://colab.research.google.com/github/raynardmiot/FormalMethodsTasting/blob/main/METAMERS_core.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Metamer math stuff
import numpy as np
from matplotlib import pyplot as plt

# All the core things
!pip install z3-solver
!pip install git+https://github.com/crrivero/FormalMethodsTasting.git#subdirectory=core
from z3 import *
from tofmcore import showSolver, list_all_solutions, draw_chessboard_with_rooks, draw_multiple_chessboards_with_rooks, rooks_output_string
from IPython.display import clear_output
clear_output()

# Metamerism Detection

Metamerisms refer to a phenomenon where two objects look like the same color under a certain light, but are in reality different colors. The two objects that cause a metamerism are referred to as metamers. We can model metamerisms mathematically as such:

Firstly, there's the **illuminant**, which shines light on our objects, and its corresponding illuminance function, $e(\lambda)$, where $\lambda$ is a wavelength. $e(\lambda)$ tells us how powerfully the illuminant emits a given wavelength, scaled between $0$ and $1$. For example, a heat lamp would have high power (and thus a larger $e(\lambda)$) for $\lambda$'s around $550$ to $700$, and close to no power for $\lambda < 500$. The sun's $e(\lambda)$ is maximal for $400 \leq \lambda \leq 500$, but also emits across the whole visible spectrum. Our example illuminant will be a simplified, simulated daylight.

When calculating metamerisms, we don't want to just know illuminance at one wavelength, however. We'd like to assess the sum total color of the illuminance across the given sensor's entire visible spectrum. So, we construct a vector $E$ that captures the power of the illuminant across the range of relevant $\lambda$. For our example, our sensor will be the human eye, and so our range of $\lambda$'s will be between $380$ and $700$, with a sampling width of $5$ nm. So, $\boldsymbol{\lambda} = [380, 385, 390, \ldots, 690, 695, 700]$.

Let's get to modeling this in code.

In [ ]:
# Wavelengths
wl_min = 380
wl_max = 700
wl = np.arange(wl_min, wl_max+1, 5) # Wavelengths 5nm apart

# Simulated daylight illuminant, using Gaussian distribution
def gaussian(wl, mu, sigma):
    return np.exp(-0.5 * ((wl - mu)/sigma)**2)

E = (
    1.2 * gaussian(wl, 600, 120) +   # warm tail
    0.8 * gaussian(wl, 450, 90)      # blue contribution
)
E /= E.max() # normalize => all values between 0 and 1

# Plotting the illuminant's SPD
plt.plot(wl, E, c='orange')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Relative Power')
plt.title('Simulated Daylight Illuminant SPD')
plt.xticks(np.arange(wl_min, wl_max+1, 50))
plt.grid(True)
plt.show()

Next, there's the **reflectant**, which is what the light illuminant's light is shining off of, i.e. our object. It has a corresponding reflectance function $r(\lambda)$, which tells us how powerfully the reflectant reflects a certain wavelength. This is very similar to illuminance, but instead of describing how something emits its own light, it describes how it reflects incoming light. Similar to our construction of $E$, we will construct a vector of reflectances $R$ over our wavelength range.

In code:

In [ ]:
# Some reflectant, nothing in particular
# Labeled r1 b/c we'll introduce another one later
R1 = (
    0.3 +
    0.4 * gaussian(wl, 520, 60) -
    0.2 * gaussian(wl, 450, 30)
)
R1 = np.clip(R1, 0, 1) # Make sure all values are between 0 and 1

# Plotting the object's reflectance SPD
plt.plot(wl, R1, c='m')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Relative Power')
plt.title("Object's Reflectance SPD")
plt.xticks(np.arange(wl_min, wl_max+1, 50))
plt.grid(True)
plt.show()

If we multiply these two objects together, we get the resultant power for a given wavelength after being emitted from its illuminant and reflected of our object. Mathmematically, let $l(\lambda) = e(\lambda)r(\lambda)$, and we can vectorize $l(\lambda)$ into $L$ in the same way we did with $E$ and $R$.

In [ ]:
L = E*R1 # pairwise multiplication, not matrix multiplication!
# We can write it with matrix multiplication (@ in numpy) as such:
# l = np.diag(e) @ r1

# Plotting the object's reflectance SPD
plt.plot(wl, E, c='orange', ls='--')
plt.plot(wl, R1, c='m', ls='--')
plt.plot(wl, L, c='c')
plt.legend(['Illuminant', 'Reflectant', 'Resultant'])
plt.xlabel('Wavelength (nm)')
plt.ylabel('Relative Power')
plt.title("Resultant SPD")
plt.xticks(np.arange(wl_min, wl_max+1, 50))
plt.grid(True)
plt.show()

We can see that the reflectant just scales down the illuminance by some factor based on the wavelength.

The last thing we have to model is the **sensor**, which will take in the resultant light from above and interpret it into a single color. To represent the sensor, we use $s(\lambda)$, which outputs a $1 \times k$ vector, where $k$ is the number of channels in the sensor and the $i$th entry in the vector represents how senstive the $i$th channel is to the given wavelength $\lambda$. For human eyes, which have three cones, $k = 3$. Again, we can "vectorize" these sensor sensitivities as we have for illuminance and reflectance, but it will generate an $n \times k$ matrix instead (let's call this matrix $\mathbf S$):

In [ ]:
# Sensor sensitivities, simplification of the human eye
S = np.stack([
    0.9 * gaussian(wl, 560, 40),  # L (red)   cone
    1.0 * gaussian(wl, 530, 35),  # M (green) cone
    0.7 * gaussian(wl, 420, 25),  # S (blue)  cone
], axis=1)
assert S.shape == (len(wl), 3)

# Plotting SPD's for the three cones
plt.xlabel('Wavelength (nm)')
plt.ylabel('Relative Power')
plt.title("Sensor Sensitivity SPD")
plt.xticks(np.arange(wl_min, wl_max+1, 50))
plt.grid(True)

colors = 'rgb'
for i, column in enumerate(S.T):
  plt.plot(wl, column, c=colors[i])
  plt.legend(['L (red)', 'M (green)', 'S (blue)'])


plt.show()

So, with these three objects (illuminance, reflectance, sensor sensitivity), we can compute the sensor's response given a certain object is being lit under a certain light:

$\boldsymbol \rho = \Delta\lambda \cdot \mathbf{S^\top} diag(E)R$

Let's walk through this. Firstly, $\Delta\lambda$ represents our sampling width of wavelengths. In our case, it's $5$nm. $diag(E)$ is simply a square matrix with the vector $\mathbf e$ on its diagonal. As for some intuition: $diag(E)R$ is simply the $L$ vector we saw earlier, which represents the resultant light after being emitted from its illuminant and reflecting off our object. It's the linear algebra way of computing `E * R1` (element-wise vector multiplication) in Python. So, simplified:

$\boldsymbol \rho = \Delta\lambda \cdot \mathbf{S^\top}L$

Lastly, we matrix multiply our sensor sensitivites with the resultant light, and we get $\rho$, the sensor's output given an object and a light source. $\rho$ is a $1 \times k$ vector, and so the output is divided into the $k$ channels of the sensor. Again, for the human eye, we have three cones so $\rho$ would be $1 \times 3$. Let's code it!

In [ ]:
delta = float(np.mean(np.diff(wl))) # Mean of the differences between the wavelengths, which should all be equal.
rho1 = delta * S.T @ L
print(rho1)

Now, for the real question:

**Given an illuminant, a sensor, and an object, how can we find another object that looks the same to the sensor as the first?** Written mathematically, here are our high-level constraints:

$\rho_{R_1} = \rho_{R_2}$, and $R_1 \neq R_2$

Since we want to focus only on the reflectances, let's rewrite our equation for $\rho$ as such:

$\rho = \mathbf A^\top R$, where $\mathbf A = \Delta\lambda\cdot diag(E)\mathbf S$

Let's expand it in code:

In [ ]:
k = 3 # n sensors
n = len(wl)

A = delta * np.diag(E) @ S # A is an n x k matrix, 65 x 3 in our case

# In Z3, represent the second reflector
R2 = [Real(f"R2_{i}") for i in range(n)]

s = Solver()

# Make sure reflectances are between 0 and 1
for R2i in R2:
  s.add(And(0 <= R2i, R2i <= 1))

# Bonus: require that the reflectance curve doesn't jump around a bunch
# for i, R2i in enumerate(R2):
#   s.add(And(0 <= R2i, R2i <= 1))
#   if i > 0:
#     s.add(And(-.05 <= R2[i-1] - R2i, R2[i-1] - R2i <= .05))

# Make sure R2's rho is equal to R1's rho
# We'll have to do it manually without linear algebra notation
s.add(
      # Just doing the matrix multiplication one by one
      Sum(A.T[0][i] * R2[i] for i in range(n)) == rho1[0],
      Sum(A.T[1][i] * R2[i] for i in range(n)) == rho1[1],
      Sum(A.T[2][i] * R2[i] for i in range(n)) == rho1[2]
)

Did we find a metamer?

In [ ]:
if s.check() == sat:
  print("Metamer found!")
else:
  print("No metamer found :(")

Let's look at it as an array:

In [ ]:
m = s.model()
R2_sol = np.zeros(len(wl))
for i, R2i in enumerate(R2):
  # Stored as a Z3 rational number reference (RatNumRef)
  reflectance_val = m.eval(R2i)

  # Get numerator and denominator of reference to reconstruct the value
  num = reflectance_val.numerator_as_long()
  denom = reflectance_val.denominator_as_long()
  R2_sol[i] = num/denom

R2_sol

Let's see what the two objects' reflectance curves look like.

In [ ]:
# Plotting the objects' reflectance SPD
plt.plot(wl, R1, c='m')
plt.plot(wl, R2_sol, c='brown')
plt.legend(['Object 1 Reflectance', 'Object 2 Reflectance'])
plt.xlabel('Wavelength (nm)')
plt.ylabel('Relative Power')
plt.title("Objects' Reflectance SPDs")
plt.xticks(np.arange(wl_min, wl_max+1, 50))
plt.grid(True)
plt.show()

A weird looking object, isn't it? But:

In [ ]:
print(f"R1 sensor values = {A.T @ R1}")
print(f"R2 sensor values = {A.T @ R2_sol}")
print(f"Sensor value difference = {(A.T @ R2_sol - A.T @ R1).round(8)}")

They'd look the same to us in daylight!